# Audio Inpainting with PlayDiffusion

Run Cell 1 (install) — wait for it to finish, then restart runtime when prompted.
Run Cell 1 again (it will skip installs and just confirm everything is ready).
Then run cells 4 through 17 sequentially.

This notebook runs the full audio inpainting pipeline on Google Colab:
1. **Setup** — clone PlayDiffusion, install matching PyTorch 2.6.0 + fairseq2 + dependencies
2. **STT + LLM** — transcribe corrupted audio with Qwen3-ASR, fix gaps with Qwen3-32B (OpenRouter)
3. **Detection** — automatic corruption detection (RMS + MFCC, plus optional Qwen2-Audio)
4. **Evaluation** — measure detection quality vs ground truth
5. **Inpainting** — use PlayDiffusion to regenerate corrupted regions

**Requirements:** Colab GPU runtime. **A100 (40 GB) recommended** if using Qwen2-Audio detection; T4 (16 GB) is enough without it.

In [ ]:
import os

# ── 1. Clone PlayDiffusion ──────────────────────────────────────────────
if not os.path.exists('/content/PlayDiffusion'):
    !git clone https://github.com/playht/PlayDiffusion.git
else:
    print("PlayDiffusion already cloned — skipping.")

# ── 2. Check whether dependencies are already installed (post-restart) ──
_need_install = True
try:
    import torch
    if torch.__version__.startswith('2.6.0'):
        import torchaudio, fairseq2
        _need_install = False
        print("Dependencies already installed. Skip to the next cell.")
except ImportError:
    pass

if _need_install:
    print("Installing dependencies — this may take a few minutes ...\n")

    # Remove Colab-default packages that conflict with PlayDiffusion
    !pip uninstall -y torch torchvision torchaudio fairseq2 fairseq2n 2>/dev/null

    # Step A: Install PyTorch 2.6.0 first
    !pip install torch==2.6.0 torchaudio==2.6.0 \
        --index-url https://download.pytorch.org/whl/cu124

    # Step B: Install fairseq2 explicitly using the PyTorch 2.6.0 + CUDA 12.4 index
    # Enforce torch==2.6.0 so fairseq2n doesn't pull a newer version
    !pip install fairseq2==0.4.4 fairseq2n==0.4.4 torch==2.6.0 torchaudio==2.6.0 \
        --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124

    # Step C: Install PlayDiffusion runtime deps
    !pip install \
        "numpy>=1.26,<2.1" \
        "scipy>=1.12" \
        "scikit-learn>=1.3" \
        nltk jiwer pydantic soundfile boto3 tqdm python-decouple \
        safetensors tokenizers librosa einops huggingface-hub unidecode \
        torchtune==0.6.1 torchao==0.11.0 torch==2.6.0 torchaudio==2.6.0
    !pip install "syllables @ git+https://github.com/playht/python-syllables.git"

    # [NEW] Ensure transformers is new enough for Qwen2-Audio-7B-Instruct
    !pip install -q "transformers==4.45.2"

    # Step D: Install Whisper (STT) + Gemini (LLM)
    !pip install openai-whisper whisper-timestamped google-generativeai torch==2.6.0 torchaudio==2.6.0

    print("\n" + "=" * 60)
    print("  RESTART THE RUNTIME NOW")
    print("  Runtime -> Restart runtime, then run the cells below.")
    print("=" * 60)

PlayDiffusion already cloned — skipping.
Dependencies already installed. Skip to the next cell.


### After running the cell above for the first time:
1. **Restart the runtime** — Runtime -> Restart runtime
2. Then continue running from the cell below (you do **not** need to re-run the install cell)

### Step 2: Verify Installation

In [ ]:
import sys
sys.path.insert(0, '/content/PlayDiffusion/src')

import torch, torchaudio
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"Torchaudio: {torchaudio.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}  (toolkit {torch.version.cuda})")

import fairseq2
print(f"fairseq2:   {fairseq2.__version__}")

from playdiffusion import PlayDiffusion, InpaintInput
print("\nPlayDiffusion imported successfully!")

Python:     3.12.13
PyTorch:    2.6.0+cu124
Torchaudio: 2.6.0+cu124
CUDA:       True  (toolkit 12.4)
fairseq2:   0.4.4

PlayDiffusion imported successfully!


### Step 3: Prepare Data
Mount Google Drive and set the input/output directories. Upload your preprocessed audio files and JSON metadata to the input directory.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Point these at the folder where your preprocessed audio + JSON metadata live.
# Option A (Google Drive — persists between sessions):

# input_dir  = "/content/drive/MyDrive/samples/preprocessed_audio"
# output_dir = "/content/drive/MyDrive/samples/inpainted_audio"

input_dir  = "/content/drive/MyDrive/samples_hindy/preprocessed"
output_dir = "/content/drive/MyDrive/samples_hindy/inpainted_audio"


os.makedirs(input_dir,  exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Input dir:  {input_dir}")
print(f"Output dir: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input dir:  /content/drive/MyDrive/samples_hindy/preprocessed
Output dir: /content/drive/MyDrive/samples_hindy/inpainted_audio


### Step 4: Load Models & Helper Functions

This cell loads:
*   Qwen3-ASR-1.7B + ForcedAligner-0.6B — for speech-to-text with native word-level timestamps (replaces Whisper)
* Qwen3-32B via OpenRouter — for LLM transcript correction

[CHANGED] ASR switched from Whisper → Qwen3-ASR (SOTA open-source, better word boundaries, native forced alignment). Whisper is kept as fallback in case Qwen3-ASR fails to load (e.g. dependency / memory issue).

Helper functions defined here (used by Steps 5 and 6):
* transcribe_with_timestamps()  — unified interface (Qwen or Whisper)
* get_corrected_text_and_original_word_times()
* has_acoustic_discontinuity()
* smooth_silence_gap()


In [ ]:
import numpy as np
import librosa
import soundfile as sf
import requests
import torch
from google.colab import userdata

# ── [NEW] Load Qwen3-ASR (primary), fall back to Whisper if it fails ──────────
USE_QWEN_ASR = False          # Set False to force Whisper instead
QWEN_ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"      # Use "Qwen3-ASR-0.6B" if OOM
QWEN_ALIGNER_ID   = "Qwen/Qwen3-ForcedAligner-0.6B"

qwen_asr_model = None
whisper_model  = None

if USE_QWEN_ASR:
    try:
        print(f"Loading {QWEN_ASR_MODEL_ID} + forced aligner ...")
        from qwen_asr import Qwen3ASRModel
        qwen_asr_model = Qwen3ASRModel.from_pretrained(
            QWEN_ASR_MODEL_ID,
            dtype=torch.bfloat16,
            device_map="cuda:0",
            max_inference_batch_size=8,
            max_new_tokens=512,
            forced_aligner=QWEN_ALIGNER_ID,
            forced_aligner_kwargs=dict(
                dtype=torch.bfloat16,
                device_map="cuda:0",
            ),
        )
        print("Qwen3-ASR loaded.")
    except Exception as e:
        print(f"Qwen3-ASR failed to load ({e}) — falling back to Whisper.")
        qwen_asr_model = None

if qwen_asr_model is None:
    print("Loading Whisper model (fallback) ...")
    import whisper
    import whisper_timestamped as whisper_ts
    whisper_model = whisper.load_model("base")

# ── Configure Qwen via OpenRouter (free LLM for transcript correction) ────────
OPENROUTER_API_KEY = None
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    print("Qwen (OpenRouter) configured.")
except Exception as e:
    print(f"Could not load OPENROUTER_API_KEY: {e}")
    print("Running without LLM correction — raw ASR transcripts will be used.")


def call_qwen_llm(transcript):
    """Sends a broken transcript to Qwen3-32B via OpenRouter for correction."""
    if OPENROUTER_API_KEY is None:
        return None
    prompt = (
        "The following text is a transcription of an audio file that had "
        "network connectivity drops. There are missing words or parts of words. "
        "Please guess what the full, logical sentence should be and return "
        "ONLY the complete, corrected sentence without any explanation.\n\n"
        f"Transcript: {transcript}"
    )
    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": "qwen/qwen3-32b:free",
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=30,
        )
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"  LLM error ({e}) — using raw transcript.")
        return None


# ── [NEW] Unified transcription function ──────────────────────────────────────
# Returns (transcript, word_timestamps) where word_timestamps is a list of
# {'word', 'start', 'end'} dicts — same format as before, regardless of
# whether Qwen3-ASR or Whisper was used underneath.
def transcribe_with_timestamps(audio_path):
    if qwen_asr_model is not None:
        # Qwen3-ASR path
        results = qwen_asr_model.transcribe(
            audio=audio_path,
            language="English",
            return_time_stamps=True,
        )
        r = results[0]
        transcript = r.text

        word_timestamps = []
        ts = getattr(r, 'time_stamps', None) or []
        # time_stamps can be a flat list of word objects or a list-of-lists;
        # handle both to be safe across package versions
        flat = ts[0] if (ts and isinstance(ts[0], (list, tuple))) else ts
        for w in flat:
            word = getattr(w, 'text', None) or (w.get('text') if isinstance(w, dict) else '')
            s    = getattr(w, 'start_time', None) or (w.get('start_time') if isinstance(w, dict) else 0.0)
            e    = getattr(w, 'end_time',   None) or (w.get('end_time')   if isinstance(w, dict) else 0.0)
            word_timestamps.append({'word': word.strip(), 'start': float(s), 'end': float(e)})
        return transcript, word_timestamps
    else:
        # Whisper fallback path
        audio     = whisper_ts.load_audio(audio_path)
        result_ts = whisper_ts.transcribe(whisper_model, audio, language="en", verbose=False)
        transcript = result_ts['text']
        word_timestamps = []
        for seg in result_ts.get('segments', []):
            for wi in seg.get('words', []):
                word_timestamps.append({
                    'word':  wi.get('text', '').strip(),
                    'start': wi['start'],
                    'end':   wi['end'],
                })
        return transcript, word_timestamps


def get_corrected_text_and_original_word_times(audio_path):
    """Transcribes audio, then corrects the transcript with the LLM."""
    print(f"  Transcribing ...")
    transcript, word_timestamps = transcribe_with_timestamps(audio_path)
    print(f"  Original:  {transcript}")

    corrected = call_qwen_llm(transcript)
    if corrected:
        print(f"  Corrected: {corrected}")
        return corrected, word_timestamps
    return transcript, word_timestamps


# ── Helper: MFCC discontinuity check at a known cut point ─────────────────────
def has_acoustic_discontinuity(audio_path, cut_time_sec, window_sec=0.05, threshold=10.0):
    y, sr      = librosa.load(audio_path, sr=None, mono=True)
    cut_sample = int(cut_time_sec * sr)
    window     = int(window_sec   * sr)
    before = y[max(0, cut_sample - window) : cut_sample]
    after  = y[cut_sample : min(len(y), cut_sample + window)]
    if len(before) < 10 or len(after) < 10:
        return False
    m_before = librosa.feature.mfcc(y=before, sr=sr, n_mfcc=13)
    m_after  = librosa.feature.mfcc(y=after,  sr=sr, n_mfcc=13)
    diff = np.linalg.norm(np.mean(m_before, axis=1) - np.mean(m_after, axis=1))
    return bool(diff > threshold)


# ── Helper: smooth a silence gap with a short crossfade ───────────────────────
def smooth_silence_gap(audio_path, start_sec, end_sec, output_path, crossfade_ms=20):
    y, sr = sf.read(audio_path)
    mono  = (y.ndim == 1)
    if mono:
        y = y[:, np.newaxis]
    s_sample = int(start_sec * sr)
    e_sample = int(end_sec   * sr)
    cf = min(int(crossfade_ms * sr / 1000), s_sample, max(0, len(y) - e_sample))
    if cf > 0:
        fade_out = np.linspace(1.0, 0.0, cf)[:, np.newaxis]
        fade_in  = np.linspace(0.0, 1.0, cf)[:, np.newaxis]
        blended  = y[s_sample - cf : s_sample] * fade_out + y[e_sample : e_sample + cf] * fade_in
        result   = np.concatenate([y[: s_sample - cf], blended, y[e_sample + cf :]])
    else:
        result = np.concatenate([y[:s_sample], y[e_sample:]])
    if mono:
        result = result[:, 0]
    sf.write(output_path, result, sr)

Loading Whisper model (fallback) ...
Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

Qwen (OpenRouter) configured.


### Step 5: Corruption Detection Module

This step adds automatic detection of corrupted audio regions. Previously, the pipeline required a JSON file with known timestamps. Now, if no events are found in the JSON, this module scans the audio automatically and produces the same (start_sec, end_sec) format.

Two complementary signal-processing methods are used:

**Method A — RMS Energy Dropout Detection**

Computes rolling loudness (dB) and flags sudden drops. Best at: full dropouts, hard silence insertions.

**Method B — MFCC Discontinuity Scan**

Computes frame-level acoustic feature distance and flags abrupt jumps.Best at: mid-word glitches, cut-and-splice artifacts.

Both methods run independently. Their results are merged into a unified list of (start_sec, end_sec) corruption candidates with padding applied.

______________________________________________________________________________

*Parameters can be tuned below if you get too many false positives (raise thresholds) or miss real glitches (lower thresholds).*

In [ ]:
import numpy as np
import librosa

# ── Method A — RMS dropout detection ──────────────────────────────────────────
RMS_WINDOW_SEC      = 0.020   # Analysis window size (20ms)
RMS_HOP_SEC         = 0.010   # Hop between windows (10ms)
RMS_DROP_THRESHOLD  = 20.0    # dB drop vs local max that flags a dropout
RMS_MIN_DURATION    = 0.020   # Ignore detections shorter than this (seconds)

# ── Method B — MFCC discontinuity scan ────────────────────────────────────────
MFCC_FRAME_SEC       = 0.025   # MFCC analysis window (25ms)
MFCC_HOP_SEC         = 0.010   # MFCC hop (10ms)
MFCC_N_COEFFS        = 13      # Number of MFCC coefficients
# [CHANGED] Replaced fixed MFCC_DIFF_THRESHOLD with an adaptive per-file threshold.
# The previous absolute value (30.0) was triggering on ~half of all frames in our
# test recordings, which collapsed everything into one giant region after merging.
MFCC_OUTLIER_K       = 6.0     # Robust z-score multiplier (median + k*MAD); raise to be stricter
MFCC_MIN_PERCENTILE  = 99.0    # Never trigger below this percentile of the diff distribution

# ── Merging and padding ───────────────────────────────────────────────────────
MERGE_GAP_SEC       = 0.05    # Fuse detections closer than this (seconds)
CONTEXT_PAD_SEC     = 0.05    # Expand each detected region by this on each side
MAX_REGION_SEC      = 2.5     # [CHANGED] was 1.0; this is just a sanity cap on absurdly large fused regions


def detect_rms_dropouts(y, sr):
    """
    [Method A] Scans the audio for sudden drops in RMS energy.
    Returns list of (start_sec, end_sec) tuples.
    """
    win = int(RMS_WINDOW_SEC * sr)
    hop = int(RMS_HOP_SEC    * sr)
    eps = 1e-10

    frames = librosa.util.frame(y, frame_length=win, hop_length=hop)
    rms    = np.sqrt(np.mean(frames ** 2, axis=0))
    rms_db = 20 * np.log10(rms + eps)
    times  = librosa.frames_to_time(np.arange(len(rms_db)), sr=sr, hop_length=hop)

    ref_frames  = max(1, int(0.3 / RMS_HOP_SEC))
    rolling_max = np.array([
        np.max(rms_db[max(0, i - ref_frames) : i + 1])
        for i in range(len(rms_db))
    ])
    dropout_mask = (rolling_max - rms_db) > RMS_DROP_THRESHOLD

    events     = []
    in_dropout = False
    start_idx  = 0
    for i, flag in enumerate(dropout_mask):
        if flag and not in_dropout:
            in_dropout = True
            start_idx  = i
        elif not flag and in_dropout:
            in_dropout = False
            if (times[i] - times[start_idx]) >= RMS_MIN_DURATION:
                events.append((times[start_idx], times[i]))
    if in_dropout and (times[-1] - times[start_idx]) >= RMS_MIN_DURATION:
        events.append((times[start_idx], times[-1]))

    return events


def detect_mfcc_discontinuities(y, sr):
    """
    [Method B, CHANGED] Scans the audio for abrupt frame-to-frame acoustic jumps.
    Uses an adaptive threshold (median + k*MAD, floored at the 99th percentile)
    so the trigger sparsity is consistent across recordings.
    Returns list of (start_sec, end_sec) tuples — zero-width here; merge_events pads.
    """
    hop   = int(MFCC_HOP_SEC   * sr)
    n_fft = int(MFCC_FRAME_SEC * sr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=MFCC_N_COEFFS,
                                  n_fft=n_fft, hop_length=hop)
    diffs = np.linalg.norm(np.diff(mfccs, axis=1), axis=0)
    times = librosa.frames_to_time(np.arange(len(diffs)), sr=sr, hop_length=hop)

    # Robust threshold: median + k*MAD, floored at the 99th percentile.
    med = float(np.median(diffs))
    mad = float(np.median(np.abs(diffs - med)) * 1.4826 + 1e-9)
    thr = max(med + MFCC_OUTLIER_K * mad,
              float(np.percentile(diffs, MFCC_MIN_PERCENTILE)))

    n_above = int((diffs > thr).sum())
    print(f"  [MFCC stats] median={med:.2f}  mad={mad:.2f}  "
          f"thr={thr:.2f}  n_above={n_above}")

    # [CHANGED] Return zero-width events; previously each event was pre-padded by
    # CONTEXT_PAD_SEC and then padded a second time inside merge_events.
    return [(float(times[i]), float(times[i])) for i, d in enumerate(diffs) if d > thr]


def merge_events_with_source(rms_events, mfcc_events, wavlm_events=None, audio_duration=None):
    """[CHANGED] Now accepts WavLM events too. Source tags: 'rms', 'mfcc', 'wavlm'."""
    wavlm_events = wavlm_events or []
    tagged = ([(s, e, "rms")   for s, e in rms_events]
              + [(s, e, "mfcc")  for s, e in mfcc_events]
              + [(s, e, "wavlm") for s, e in wavlm_events])
    if not tagged:
        return []
    tagged.sort(key=lambda x: x[0])

    merged = [[tagged[0][0], tagged[0][1], {tagged[0][2]}]]
    for s, e, src in tagged[1:]:
        if s - merged[-1][1] <= MERGE_GAP_SEC:
            merged[-1][1] = max(merged[-1][1], e)
            merged[-1][2].add(src)
        else:
            merged.append([s, e, {src}])

    result = []
    for s, e, srcs in merged:
        s = max(0.0, s - CONTEXT_PAD_SEC)
        e = e + CONTEXT_PAD_SEC
        if audio_duration:
            e = min(audio_duration, e)
        result.append((s, e, srcs))

    kept, dropped = [], []
    for s, e, srcs in result:
        (kept if (e - s) <= MAX_REGION_SEC else dropped).append((s, e, srcs))
    if dropped:
        print(f"  [WARN] dropped {len(dropped)} oversized region(s) (>{MAX_REGION_SEC}s)")

    return kept


def detect_corrupted_regions(audio_path, verbose=True, mode="any"):
    """
    [CHANGED] Now runs WavLM in parallel with RMS+MFCC when available.
    Modes: 'any', 'both' (RMS AND MFCC), 'rms', 'mfcc', 'wavlm',
           'wavlm+rms' (WavLM AND RMS — the new precision-recall sweet spot to test).
    """
    y, sr    = librosa.load(audio_path, sr=None, mono=True)
    duration = len(y) / sr

    rms_events   = detect_rms_dropouts(y, sr)
    mfcc_events  = detect_mfcc_discontinuities(y, sr)
    # WavLM is optional — call only if Step 5c loaded the model
    try:
        wavlm_events = detect_wavlm_discontinuities(audio_path, verbose=verbose)
    except NameError:
        wavlm_events = []

    if verbose:
        print(f"  [Method A — RMS]   {len(rms_events)} candidate(s)")
        print(f"  [Method B — MFCC]  {len(mfcc_events)} candidate(s)")
        print(f"  [Method C — WavLM] {len(wavlm_events)} candidate(s)")

    merged = merge_events_with_source(rms_events, mfcc_events, wavlm_events,
                                       audio_duration=duration)

    if mode == "both":
        merged = [(s, e, srcs) for s, e, srcs in merged if {"rms", "mfcc"}.issubset(srcs)]
    elif mode == "rms":
        merged = [(s, e, srcs) for s, e, srcs in merged if "rms" in srcs]
    elif mode == "mfcc":
        merged = [(s, e, srcs) for s, e, srcs in merged if "mfcc" in srcs]
    elif mode == "wavlm":
        merged = [(s, e, srcs) for s, e, srcs in merged if "wavlm" in srcs]
    elif mode == "wavlm+rms":
        merged = [(s, e, srcs) for s, e, srcs in merged if {"wavlm", "rms"}.issubset(srcs)]
    # mode == "any" keeps everything

    if verbose:
        print(f"  [Merged, mode={mode}] {len(merged)} region(s):")
        for s, e, srcs in merged:
            print(f"                    {s:.3f}s – {e:.3f}s  [{'+'.join(sorted(srcs))}]")

    return [{"start_sec": s, "end_sec": e, "source": "+".join(sorted(srcs))}
            for s, e, srcs in merged]


print("Step 5 ready: Corruption detection module loaded.")
print(f"  RMS drop threshold:  {RMS_DROP_THRESHOLD} dB")
print(f"  MFCC adaptive:       median + {MFCC_OUTLIER_K}*MAD, floor=p{MFCC_MIN_PERCENTILE}")
print(f"  Merge gap:           {MERGE_GAP_SEC}s")
print(f"  Max region:          {MAX_REGION_SEC}s (sanity cap)")

Step 5 ready: Corruption detection module loaded.
  RMS drop threshold:  20.0 dB
  MFCC adaptive:       median + 6.0*MAD, floor=p99.0
  Merge gap:           0.05s
  Max region:          2.5s (sanity cap)


### Step 5b: Qwen2-Audio Detection (Optional, A100 recommended)

[NEW] Uses Qwen2-Audio-7B-Instruct — a multimodal LLM that accepts audio as input — to *reason* about the recording and identify dropout regions.

Unlike RMS+MFCC (pure signal processing) or Qwen3-ASR (pure transcription) this model simultaneously understands the ACOUSTIC content AND the LINGUISTIC context, so it can detect the "so__ wheat" case where ASR confidently misreads a mid-word glitch as a clean word.

Memory:    Qwen2-Audio-7B in bfloat16 uses ~14 GB of GPU RAM.
Recommended runtime: A100 (40 GB) with High-RAM enabled.
If it fails to load, set USE_QWEN_AUDIO = False below to skip it.

Output format: same as detect_corrupted_regions() — a list of {"start_sec", "end_sec"} dicts — so it plugs into the same evaluation machinery in Step 7.

In [ ]:
# [CHANGED] Qwen2-Audio disabled — confirmed wrong tool for short-dropout detection.
# It hallucinated timestamps under open-ended prompts and returned empty under
# structured prompts. Keeping the function stub so Step 7a code paths still work.
USE_QWEN_AUDIO   = False
qwen_audio_model = None

def detect_with_qwen_audio(audio_path, verbose=True, debug=False):
    return []

print("Step 5b: Qwen2-Audio disabled (low-quality outputs on this task).")

Step 5b: Qwen2-Audio disabled (low-quality outputs on this task).


In [ ]:
# ── Step 5c: WavLM Frame-Embedding Discontinuity Detection [NEW] ─────────────
#
# Computes WavLM frame embeddings (50Hz / 20ms hops) and flags frames where the
# cosine distance to the previous frame is an outlier. Catches phonetic
# discontinuities that MFCC misses — specifically the "mid-word splice" case
# where spectral shape stays similar but phonetic content jumps abruptly.
#
# Model: microsoft/wavlm-base-plus (95M, ~360MB). Loads to GPU, inference only.
# ─────────────────────────────────────────────────────────────────────────────

import torch
import numpy as np
import librosa

USE_WAVLM = True

wavlm_model     = None
wavlm_processor = None

if USE_WAVLM:
    try:
        print("Loading WavLM-base-plus ...")
        from transformers import AutoFeatureExtractor, WavLMModel
        wavlm_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
        wavlm_model     = WavLMModel.from_pretrained(
            "microsoft/wavlm-base-plus",
            torch_dtype=torch.float32,   # WavLM is small enough; fp32 keeps cosine distances stable
        ).to("cuda:0").eval()
        print("WavLM loaded.")
    except Exception as e:
        print(f"WavLM failed to load ({e}) — skipping this method.")
        wavlm_model = None


# ── Detection params ──────────────────────────────────────────────────────────
WAVLM_OUTLIER_K      = 3.0    # [CHANGED] was 5.0; cosine MAD is large so 5*MAD over-shot every distance
WAVLM_MIN_PERCENTILE = 97.0   # [CHANGED] was 99.0; ~p99 was 4-5 frames on 9s clips, too sparse to gate on
WAVLM_HOP_SEC        = 0.020  # WavLM-base outputs at 50Hz (one frame per 20ms)


def _wavlm_frame_embeddings(audio_path):
    """
    Returns (embeddings [T, D], frame_times [T]) for the audio.
    WavLM-base outputs 768-dim embeddings at 50Hz.
    """
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = wavlm_processor(audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(wavlm_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = wavlm_model(**inputs)
    # last_hidden_state: [1, T, 768]
    emb = out.last_hidden_state[0].float().cpu().numpy()
    times = np.arange(emb.shape[0]) * WAVLM_HOP_SEC
    return emb, times


def detect_wavlm_discontinuities(audio_path, verbose=True):
    """
    [NEW] Frame-to-frame cosine distance on WavLM embeddings, flagged with
    the same robust adaptive threshold we use for MFCC. Returns zero-width
    events; merge_events_with_source pads them.
    """
    if wavlm_model is None:
        return []

    emb, times = _wavlm_frame_embeddings(audio_path)

    # Cosine distance between consecutive frames
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9
    unit  = emb / norms
    cos_sim = np.sum(unit[1:] * unit[:-1], axis=1)
    dists   = 1.0 - cos_sim                     # length T-1
    diff_times = times[1:]

    med = float(np.median(dists))
    mad = float(np.median(np.abs(dists - med)) * 1.4826 + 1e-9)
    thr = max(med + WAVLM_OUTLIER_K * mad,
              float(np.percentile(dists, WAVLM_MIN_PERCENTILE)))

    if verbose:
      p95 = float(np.percentile(dists, 95))
      p99 = float(np.percentile(dists, 99))
      p_max = float(dists.max())
      print(f"  [WavLM stats] median={med:.4f}  mad={mad:.4f}  "
            f"p95={p95:.4f}  p99={p99:.4f}  max={p_max:.4f}  "
            f"thr={thr:.4f}  n_above={int((dists > thr).sum())}")

    return [(float(diff_times[i]), float(diff_times[i]))
            for i, d in enumerate(dists) if d > thr]


print("Step 5c ready.")
print(f"  WavLM available: {wavlm_model is not None}")

Loading WavLM-base-plus ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


WavLM loaded.
Step 5c ready.
  WavLM available: True


### Step 6: Evaluation Helpers

[NEW] When JSON events are available, we treat them as GROUND TRUTH and run the auto-detection (Step 5) in parallel to measure how well detection is doing on real data.

Metrics computed per file and aggregated at the end of Step 7:
* Recall    = fraction of JSON events that were detected
* Precision = fraction of detected events that match a real JSON event
* Mean IoU  = average Intersection-over-Union for matched pairs

An event is "matched" if its time range overlaps with a JSON event with IoU >= MATCH_IOU_THRESHOLD.

The results of Step 7 still use the JSON events for actual inpainting (so the repair is as correct as possible); detection runs purely for measurement.


In [ ]:
import numpy as np

# Lowered from 0.5 — dropouts are short, so even small timing offsets can
# push a valid match below 0.5 IoU. 0.3 is more forgiving while still
# enforcing meaningful overlap.
MATCH_IOU_THRESHOLD = 0.3


def _iou(a, b):
    """Intersection-over-Union of two (start, end) intervals."""
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.0


def evaluate_detection(gt_events, detected_events):
    """
    Compares ground-truth JSON events to auto-detected events.
    Returns a dict with counts and recall / precision / mean_iou.

    Matching is greedy: for each GT event, pick the unmatched detection
    with the highest IoU. If that IoU >= MATCH_IOU_THRESHOLD, count as matched.
    """
    gt  = [(e["start_sec"], e["end_sec"]) for e in gt_events]
    det = [(e["start_sec"], e["end_sec"]) for e in detected_events]

    matched_gt  = set()
    matched_det = set()
    iou_values  = []

    for i, g in enumerate(gt):
        best_j, best_iou = -1, 0.0
        for j, d in enumerate(det):
            if j in matched_det:
                continue
            iou = _iou(g, d)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= MATCH_IOU_THRESHOLD:
            matched_gt.add(i)
            matched_det.add(best_j)
            iou_values.append(best_iou)

    recall    = len(matched_gt)  / len(gt)  if gt  else 1.0
    precision = len(matched_det) / len(det) if det else 1.0
    mean_iou  = float(np.mean(iou_values)) if iou_values else 0.0

    return {
        "n_gt":       len(gt),
        "n_detected": len(det),
        "n_matched":  len(matched_gt),
        "recall":     recall,
        "precision":  precision,
        "mean_iou":   mean_iou,
    }


def print_aggregate_metrics(totals):
    """Pretty-print the running totals collected across all processed files."""
    if totals["n_gt"] == 0:
        print("\nNo ground-truth events available — skipping aggregate metrics.")
        return

    overall_recall    = totals["n_matched"]  / totals["n_gt"]
    overall_precision = (totals["n_matched"] / totals["n_detected"]
                          if totals["n_detected"] else 0.0)
    overall_mean_iou  = (totals["iou_sum"] / totals["iou_count"]
                          if totals["iou_count"] else 0.0)

    print(f"\n{'=' * 60}")
    print("AGGREGATE DETECTION METRICS")
    print(f"{'=' * 60}")
    print(f"  Total GT events:       {totals['n_gt']}")
    print(f"  Total detected events: {totals['n_detected']}")
    print(f"  Total matched:         {totals['n_matched']}")
    print(f"  Overall recall:        {overall_recall:.2%}")
    print(f"  Overall precision:     {overall_precision:.2%}")
    print(f"  Mean IoU (matched):    {overall_mean_iou:.3f}")


print("Step 6 ready: evaluation helpers loaded.")
print(f"  IoU match threshold: {MATCH_IOU_THRESHOLD}")

Step 6 ready: evaluation helpers loaded.
  IoU match threshold: 0.3


### Step 7a: Detection Evaluation Only (No Inpainting)

[NEW] Lightweight evaluation cell that runs ONLY the detection methods against the JSON ground truth — no PlayDiffusion, no LLM correction, no temp files. Use this to iterate quickly on detection thresholds or on the Qwen2-Audio prompt.

Toggle the `DEBUG_SIGNAL` and `DEBUG_QWEN` flags inside the cell to see intermediate candidate counts and the raw Qwen2-Audio responses.

### Step 7: Run the Full Inpainting Pipeline

Loads PlayDiffusion and processes every sample in input_dir.
For each audio file it:

1. Transcribes with Qwen3-ASR (or Whisper fallback) and corrects with Qwen LLM — all via the helpers defined in Step 4
2. Runs automatic corruption detection (Step 5 + optional Step 5b)
3. If JSON events exist: evaluates BOTH detection methods vs ground truth and uses the JSON events for actual inpainting. If no JSON events: uses the signal-processing detected events
4. Iterates over the chosen events:
   - Re-transcribes the current (partially repaired) temp file
   - Snaps event boundaries to real word boundaries
   - Silence gap → crossfade smoothing (no PlayDiffusion)
   - Acoustic glitch invisible to ASR → forces text diff
   - Normal case → calls PlayDiffusion to regenerate the region
5. Saves the repaired file as `<name>_inpainted.wav`
6. After all files: prints aggregate detection metrics for BOTH methods (signal-processing vs Qwen2-Audio) side by side

**[CHANGED from previous version]**
- Also evaluates Qwen2-Audio detection in parallel when available
- Aggregate summary prints both methods' metrics for comparison

In [ ]:
import sys, os, json, glob
sys.path.insert(0, '/content/PlayDiffusion/src')

import numpy as np
import soundfile as sf
import librosa
import torchaudio
from playdiffusion import PlayDiffusion, InpaintInput
import scipy.io.wavfile as wav

# [NEW] Optional dependency for STOI; install if missing
try:
    from pystoi import stoi as compute_stoi
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pystoi"], check=True)
    from pystoi import stoi as compute_stoi


# ── Load PlayDiffusion ────────────────────────────────────────────────────────
print("Loading PlayDiffusion model ...")
inpainter = PlayDiffusion()
print("Model loaded.\n")


# ── [NEW] Configuration ───────────────────────────────────────────────────────
USE_DETECTED_EVENTS = False              # True: test detection. False: use GT JSON.
DETECTION_MODE      = "both"             # "any" | "both" | "rms" | "mfcc" | "wavlm" | "wavlm+rms"
CLEAN_AUDIO_DIR     = "/content/drive/MyDrive/samples/raw_audio"
PAD_SEC             = 0.1               # silence padding for PlayDiffusion edge cases


# ── [NEW] Audio similarity helpers ────────────────────────────────────────────
def _align_length(a, b):
    """Trim or zero-pad b to match a's length."""
    if len(b) < len(a):
        b = np.concatenate([b, np.zeros(len(a) - len(b), dtype=b.dtype)])
    elif len(b) > len(a):
        b = b[:len(a)]
    return b


def _to_mono_16k(y, sr):
    """Both metrics need mono 16kHz."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != 16000:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000)
    return y.astype(np.float32)


def audio_similarity(reference_path, candidate_path):
    """
    [NEW] Compares candidate to reference.
    Returns dict with mel_mse (lower=better) and stoi (0-1, higher=better).
    """
    ref, sr_ref = sf.read(reference_path)
    cand, sr_cand = sf.read(candidate_path)

    ref  = _to_mono_16k(ref, sr_ref)
    cand = _to_mono_16k(cand, sr_cand)
    cand = _align_length(ref, cand)

    # Mel-spectrogram MSE
    mel_kwargs = dict(sr=16000, n_fft=512, hop_length=160, n_mels=64)
    mel_ref  = librosa.power_to_db(librosa.feature.melspectrogram(y=ref,  **mel_kwargs))
    mel_cand = librosa.power_to_db(librosa.feature.melspectrogram(y=cand, **mel_kwargs))
    mel_mse  = float(np.mean((mel_ref - mel_cand) ** 2))

    # STOI (extended STOI handles various distortions better)
    try:
        stoi_score = float(compute_stoi(ref, cand, 16000, extended=True))
    except Exception as e:
        print(f"    STOI computation failed: {e}")
        stoi_score = float("nan")

    return {"mel_mse": mel_mse, "stoi": stoi_score}


# ── Running totals ────────────────────────────────────────────────────────────
eval_totals = {"n_gt": 0, "n_detected": 0, "n_matched": 0,
               "iou_sum": 0.0, "iou_count": 0}

# [NEW] Per-sample audio quality results
quality_results = []


# ── Main loop ─────────────────────────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(input_dir, "*.json")))

if not json_files:
    print(f"No JSON metadata files found in {input_dir}")
    print("Upload your preprocessed audio files + JSON metadata first.")

for meta_file in json_files:
    with open(meta_file, 'r') as f:
        metadata = json.load(f)

    audio_path = os.path.join(input_dir, os.path.basename(metadata.get("output", "")))
    base       = os.path.splitext(os.path.basename(audio_path))[0]
    clean_path = os.path.join(CLEAN_AUDIO_DIR, f"{base}.wav")

    print(f"{'=' * 60}")
    print(f"Processing: {os.path.basename(audio_path)}")

    if not os.path.exists(audio_path):
        print("  Audio file not found — skipping.\n")
        continue

    # ── Transcribe and correct ────────────────────────────────────────────────
    corrected_text, original_word_timestamps = get_corrected_text_and_original_word_times(audio_path)

    # ── Run detection + evaluate against GT ───────────────────────────────────
    gt_events = metadata.get("events", [])

    print(f"  Running detection (mode='{DETECTION_MODE}') ...")
    detected_events = detect_corrupted_regions(audio_path, verbose=True, mode=DETECTION_MODE)

    if gt_events:
        metrics = evaluate_detection(gt_events, detected_events)
        print(f"  [Detection] GT: {metrics['n_gt']}  "
              f"Detected: {metrics['n_detected']}  Matched: {metrics['n_matched']}")
        print(f"  [Detection] Recall: {metrics['recall']:.2%}  "
              f"Precision: {metrics['precision']:.2%}  "
              f"Mean IoU: {metrics['mean_iou']:.3f}")

        eval_totals["n_gt"]       += metrics["n_gt"]
        eval_totals["n_detected"] += metrics["n_detected"]
        eval_totals["n_matched"]  += metrics["n_matched"]
        eval_totals["iou_sum"]    += metrics["mean_iou"] * metrics["n_matched"]
        eval_totals["iou_count"]  += metrics["n_matched"]

    # [CHANGED] Choose which events to inpaint based on flag
    if USE_DETECTED_EVENTS:
        events = detected_events
        print(f"  Using {len(events)} DETECTED region(s) for inpainting.")
    else:
        events = gt_events if gt_events else detected_events
        print(f"  Using {len(events)} GT region(s) for inpainting.")

    # [NEW] Deduplicate overlapping/adjacent events before inpainting.
    # Without this, RMS over-firing produces 14 overlapping events that all
    # cover the same region — PlayDiffusion runs 14 times in a row, each
    # pass drifting further from the original audio.
    DEDUP_GAP_SEC = 0.30
    if len(events) > 1:
        sorted_events = sorted(events, key=lambda e: e["start_sec"])
        deduped = [dict(sorted_events[0])]
        for e in sorted_events[1:]:
            if e["start_sec"] - deduped[-1]["end_sec"] <= DEDUP_GAP_SEC:
                deduped[-1]["end_sec"] = max(deduped[-1]["end_sec"], e["end_sec"])
            else:
                deduped.append(dict(e))
        if len(deduped) < len(events):
            print(f"  [DEDUP] {len(events)} events → {len(deduped)} after merging "
                  f"overlapping/adjacent regions (gap ≤ {DEDUP_GAP_SEC}s)")
        events = deduped

    if not events:
        print("  No events to inpaint — copying input as output.\n")
        out_file = os.path.join(output_dir, f"{base}_inpainted.wav")
        y, sr = sf.read(audio_path)
        sf.write(out_file, y, sr)
    else:
        temp_path = os.path.join(output_dir, f"{base}_temp.wav")

        # ── Write padded temp file ────────────────────────────────────────────
        y, sr = sf.read(audio_path)
        pad   = np.zeros(int(PAD_SEC * sr), dtype=y.dtype)
        if y.ndim == 1:
            y_padded = np.concatenate([pad, y, pad])
        else:
            pad2d    = np.zeros((int(PAD_SEC * sr), y.shape[1]), dtype=y.dtype)
            y_padded = np.concatenate([pad2d, y, pad2d])
        sf.write(temp_path, y_padded, sr)

        events_padded = [
            {"start_sec": e["start_sec"] + PAD_SEC, "end_sec": e["end_sec"] + PAD_SEC}
            for e in events
        ]

        # ── Inpaint each event (unchanged from before) ────────────────────────
        for i, event in enumerate(events_padded):
            original_start_sec = event.get("start_sec")
            original_end_sec   = event.get("end_sec")

            current_transcript, current_word_timestamps = transcribe_with_timestamps(temp_path)

            overlapping = [
                (idx, w) for idx, w in enumerate(current_word_timestamps)
                if w['start'] < original_end_sec and w['end'] > original_start_sec
            ]

            if overlapping:
                ov_indices, ov_words = zip(*overlapping)
                adjusted_start_sec = min(w['start'] for w in ov_words)
                adjusted_end_sec   = max(w['end']   for w in ov_words)
                in_silence = False
            else:
                adjusted_start_sec = original_start_sec
                adjusted_end_sec   = original_end_sec
                in_silence = True

            print(f"  Event {i+1}: original cut  {original_start_sec:.3f}s – {original_end_sec:.3f}s")
            print(f"             adjusted       {adjusted_start_sec:.3f}s – {adjusted_end_sec:.3f}s")

            if in_silence:
                print("  → Silence gap: applying crossfade smoothing.")
                smooth_silence_gap(temp_path, adjusted_start_sec, adjusted_end_sec, temp_path)
                continue

            if has_acoustic_discontinuity(temp_path, original_start_sec):
                keep_indices   = set(range(len(current_word_timestamps))) - set(ov_indices)
                modified_input = ' '.join(
                    current_word_timestamps[idx]['word'] for idx in sorted(keep_indices)
                )
                print(f"  → Acoustic artifact detected — forcing text diff.")
                print(f"    Modified input_text: {modified_input}")
                input_text_for_inpaint = modified_input
            else:
                input_text_for_inpaint = current_transcript

            try:
                request = InpaintInput(
                    audio=temp_path,
                    input_text=input_text_for_inpaint,
                    output_text=corrected_text,
                    start_time=adjusted_start_sec,
                    end_time=adjusted_end_sec,
                    input_word_times=current_word_timestamps,
                )
                result_audio = inpainter.inpaint(request)
                sample_rate, audio_data = result_audio
                wav.write(temp_path, sample_rate, audio_data)

            except Exception as e:
                print(f"  FAILED: {e}")

        # ── Trim padding and save final output ────────────────────────────────
        y_final, sr_final = sf.read(temp_path)
        pad_samples = int(PAD_SEC * sr_final)
        if y_final.ndim == 1:
            y_final = y_final[pad_samples : len(y_final) - pad_samples]
        else:
            y_final = y_final[pad_samples : len(y_final) - pad_samples, :]

        out_file = os.path.join(output_dir, f"{base}_inpainted.wav")
        sf.write(out_file, y_final, sr_final)
        os.remove(temp_path)
        print(f"  Saved: {out_file}")

    # ── [NEW] Audio quality eval against clean original ───────────────────────
    if os.path.exists(clean_path):
        print(f"  Evaluating vs clean original ...")
        sim_corrupt   = audio_similarity(clean_path, audio_path)
        sim_inpainted = audio_similarity(clean_path, out_file)
        print(f"    Corrupted   vs clean: mel_mse={sim_corrupt['mel_mse']:.2f}  "
              f"stoi={sim_corrupt['stoi']:.3f}")
        print(f"    Inpainted   vs clean: mel_mse={sim_inpainted['mel_mse']:.2f}  "
              f"stoi={sim_inpainted['stoi']:.3f}")
        print(f"    Lift:                 mel_mse={sim_corrupt['mel_mse'] - sim_inpainted['mel_mse']:+.2f}  "
              f"stoi={sim_inpainted['stoi'] - sim_corrupt['stoi']:+.3f}")

        quality_results.append({
            "sample":    base,
            "n_events":  len(events),
            "mel_corrupt":   sim_corrupt["mel_mse"],
            "mel_inpainted": sim_inpainted["mel_mse"],
            "stoi_corrupt":   sim_corrupt["stoi"],
            "stoi_inpainted": sim_inpainted["stoi"],
        })
    else:
        print(f"  [WARN] Clean original not found at {clean_path} — skipping quality eval.")

    print()


# ── Aggregate detection metrics ───────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"AGGREGATE DETECTION METRICS — mode='{DETECTION_MODE}', "
      f"using {'DETECTED' if USE_DETECTED_EVENTS else 'GT'} events")
print(f"{'=' * 60}")
print_aggregate_metrics(eval_totals)


# ── [NEW] Aggregate audio quality ─────────────────────────────────────────────
if quality_results:
    print(f"\n{'=' * 60}")
    print("AGGREGATE AUDIO QUALITY (vs clean original)")
    print(f"{'=' * 60}")
    print(f"{'sample':<12} {'#ev':>4} {'mel_corr':>10} {'mel_inp':>10} {'mel_lift':>10} "
          f"{'stoi_corr':>10} {'stoi_inp':>10} {'stoi_lift':>10}")
    for r in quality_results:
        mel_lift  = r["mel_corrupt"] - r["mel_inpainted"]
        stoi_lift = r["stoi_inpainted"] - r["stoi_corrupt"]
        marker    = "✓" if (mel_lift > 0 and stoi_lift > 0) else (
                    "✗" if (mel_lift < 0 and stoi_lift < 0) else "~")
        print(f"{r['sample']:<12} {r['n_events']:>4} "
              f"{r['mel_corrupt']:>10.2f} {r['mel_inpainted']:>10.2f} {mel_lift:>+10.2f} "
              f"{r['stoi_corrupt']:>10.3f} {r['stoi_inpainted']:>10.3f} {stoi_lift:>+10.3f}  {marker}")

    n = len(quality_results)
    avg_mel_lift  = sum(r["mel_corrupt"] - r["mel_inpainted"] for r in quality_results) / n
    avg_stoi_lift = sum(r["stoi_inpainted"] - r["stoi_corrupt"] for r in quality_results) / n
    n_better      = sum(1 for r in quality_results
                        if r["mel_corrupt"] > r["mel_inpainted"]
                        and r["stoi_inpainted"] > r["stoi_corrupt"])
    n_worse       = sum(1 for r in quality_results
                        if r["mel_corrupt"] < r["mel_inpainted"]
                        and r["stoi_inpainted"] < r["stoi_corrupt"])
    print(f"\n  Average mel_mse lift (corrupt - inpainted, higher=better): {avg_mel_lift:+.2f}")
    print(f"  Average STOI lift    (inpainted - corrupt, higher=better):  {avg_stoi_lift:+.3f}")
    print(f"  Samples improved on both metrics: {n_better}/{n}")
    print(f"  Samples worse on both metrics:    {n_worse}/{n}")

Loading PlayDiffusion model ...
Loading tokenizer
Loading speech tokenizer
Using vocoder checkpoint: /root/.cache/huggingface/hub/models--PlayHT--inpainter/snapshots/9c5623830cb9c4632fd4d2f53b8e6c6ec27f4a1c/v090_g_01105000


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
Using inpainter checkpoint: /root/.cache/huggingface/hub/models--PlayHT--inpainter/snapshots/9c5623830cb9c4632fd4d2f53b8e6c6ec27f4a1c/last_250k_fixed.pkl
Model loaded.

Processing: B01_01.wav
  Transcribing ...


100%|██████████| 1102/1102 [00:01<00:00, 994.33frames/s]


  Original:   Hi, my name is Patricia Brown. I lost my debit card. Can you send me a new one? My debit card. Yes, my debit card. No, that was going to be it. Bye.
  LLM error ('choices') — using raw transcript.
  Running detection (mode='both') ...
  [MFCC stats] median=30.67  mad=17.89  thr=159.05  n_above=12
  [WavLM stats] median=0.1596  mad=0.1221  p95=0.3950  p99=0.4705  max=0.6114  thr=0.5259  n_above=3
  [Method A — RMS]   27 candidate(s)
  [Method B — MFCC]  12 candidate(s)
  [Method C — WavLM] 3 candidate(s)
  [Merged, mode=both] 8 region(s):
                    1.200s – 1.420s  [mfcc+rms]
                    1.450s – 1.660s  [mfcc+rms]
                    3.210s – 3.600s  [mfcc+rms]
                    4.620s – 4.790s  [mfcc+rms]
                    4.990s – 5.400s  [mfcc+rms]
                    6.350s – 6.750s  [mfcc+rms]
                    7.720s – 8.270s  [mfcc+rms+wavlm]
                    8.560s – 8.940s  [mfcc+rms]
  [Detection] GT: 9  Detected: 8  Matched: 2
  [Dete

...

  Event 5: original cut  6.533s – 6.645s
             adjusted       6.640s – 6.980s
  → Acoustic artifact detected — forcing text diff.
    Modified input_text: Hi, my name is Mary Davis. What are your local branch hours? What are the local branch hours? No, thank you. too. Thank you. Bye bye
Input: output_text=' Hi, my name is Mary Davis. What are your local branch hours? Uh-huh. What are the local branch hours? No, thank you. You too. Thank you. Bye-bye.' num_steps=30 init_temp=1.0 init_diversity=1.0 guidance=0.5 rescale=0.7 topk=25 audio_token_syllable_ratio=None audio='/content/drive/MyDrive/samples_hindy/inpainted_audio/B08_05_temp.wav' input_text='Hi, my name is Mary Davis. What are your local branch hours? What are the local branch hours? No, thank you. too. Thank you. Bye bye' input_word_times=[{'word': 'Hi,', 'start': np.float64(0.14), 'end': np.float64(0.28)}, {'word': 'my', 'start': np.float64(0.42), 'end': np.float64(0.46)}, {'word': 'name', 'start': np.float64(0.46), 'end

100%|██████████| 864/864 [00:00<00:00, 1737.58frames/s]


  Event 6: original cut  6.980s – 7.081s
             adjusted       6.920s – 7.080s
  → Acoustic artifact detected — forcing text diff.
    Modified input_text: Hi, my name is Mary Davis. What are your local branch hours? What are the local branch hours? No, thank you. You Thank you. Bye bye
Input: output_text=' Hi, my name is Mary Davis. What are your local branch hours? Uh-huh. What are the local branch hours? No, thank you. You too. Thank you. Bye-bye.' num_steps=30 init_temp=1.0 init_diversity=1.0 guidance=0.5 rescale=0.7 topk=25 audio_token_syllable_ratio=None audio='/content/drive/MyDrive/samples_hindy/inpainted_audio/B08_05_temp.wav' input_text='Hi, my name is Mary Davis. What are your local branch hours? What are the local branch hours? No, thank you. You Thank you. Bye bye' input_word_times=[{'word': 'Hi,', 'start': np.float64(0.16), 'end': np.float64(0.28)}, {'word': 'my', 'start': np.float64(0.42), 'end': np.float64(0.46)}, {'word': 'name', 'start': np.float64(0.46), 'end':

100%|██████████| 892/892 [00:00<00:00, 1222.53frames/s]


  Event 7: original cut  8.573s – 8.576s
             adjusted       8.000s – 8.660s
  → Acoustic artifact detected — forcing text diff.
    Modified input_text: Hi, my name is Mary Davis. What are your local branch hours? Uh-huh. What are the local branch hours? No thank you. You too. Thank you. bye.
Input: output_text=' Hi, my name is Mary Davis. What are your local branch hours? Uh-huh. What are the local branch hours? No, thank you. You too. Thank you. Bye-bye.' num_steps=30 init_temp=1.0 init_diversity=1.0 guidance=0.5 rescale=0.7 topk=25 audio_token_syllable_ratio=None audio='/content/drive/MyDrive/samples_hindy/inpainted_audio/B08_05_temp.wav' input_text='Hi, my name is Mary Davis. What are your local branch hours? Uh-huh. What are the local branch hours? No thank you. You too. Thank you. bye.' input_word_times=[{'word': 'Hi,', 'start': np.float64(0.16), 'end': np.float64(0.28)}, {'word': 'my', 'start': np.float64(0.42), 'end': np.float64(0.46)}, {'word': 'name', 'start': np.flo

In [ ]:
# This clears the output folder !
# import glob, os

# for f in glob.glob(os.path.join(output_dir, "*")):
#     os.remove(f)

# print("Output folder cleared.")

Output folder cleared.
